In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

In [2]:
# ---------------------------------------------------------
# 1. 데이터 생성 (연구사님 제공 로직 기반)
# ---------------------------------------------------------
def generate_virtual_data(n_samples=5000):
    np.random.seed(2026)
    
    # 기여율(Label): Dirichlet 분포 (5개 오염원: Manure, Sewage, Fertilizer, Soil, Rain)
    props = np.random.dirichlet([1, 1, 1, 1, 1], n_samples)
    
    # 오염원별 특성 평균 (d15N, d18O, Cl, TOC)
    src_mean = np.array([
        [15.0, 5.0, 100, 20],   # Manure
        [10.0, 2.0, 80, 10],    # Sewage
        [-2.0, 15.0, 40, 2],    # Fertilizer
        [5.0, 5.0, 10, 5],      # Soil
        [-5.0, -10.0, 2, 1]     # Rain
    ])
    
    # 혼합물 생성 (Linear Mixing) + Gaussian Noise
    noise = np.random.normal(0, 1, (n_samples, 4))
    features = np.dot(props, src_mean) + noise
    
    return features, props

# 데이터 준비
X, y = generate_virtual_data(5000)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 스케일링 (Deep Learning 필수)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# ---------------------------------------------------------
# 2. SMOTE 구현 (특정 오염원 지배 시료 증폭용)
# ---------------------------------------------------------
# 예: Rain(5번째 열) 기여율이 0.5 이상인 희귀 케이스 증폭
is_rain_dominant = (y_train[:, 4] >= 0.5).astype(int)
sm = SMOTE(random_state=42)
X_train_res, _ = sm.fit_resample(X_train_scaled, is_rain_dominant)

# 주의: SMOTE는 X값만 생성하므로, 기여율(y)은 재샘플링된 인덱스에 맞춰 재구성하거나 
# 분류 모델 데이터 불균형 해소용으로 주로 활용합니다.

In [4]:
# ---------------------------------------------------------
# 3. 모델 구축 (DNN, CNN, LSTM)
# ---------------------------------------------------------

# [DNN] 다층 퍼셉트론
def build_dnn(input_dim, output_dim):
    model = models.Sequential([
        layers.Dense(64, activation='relu', input_shape=(input_dim,)),
        layers.Dense(32, activation='relu'),
        layers.Dense(output_dim, activation='softmax') # 합계 1 보장
    ])
    model.compile(optimizer='adam', loss='kl_divergence', metrics=['mae'])
    return model

# [CNN] 1D-Convolution (특성 간 지역적 패턴 학습)
def build_cnn(input_dim, output_dim):
    model = models.Sequential([
        layers.Reshape((input_dim, 1), input_shape=(input_dim,)),
        layers.Conv1D(32, kernel_size=2, activation='relu'),
        layers.Flatten(),
        layers.Dense(16, activation='relu'),
        layers.Dense(output_dim, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='kl_divergence')
    return model

# [LSTM] 시계열/순차적 특성 고려
def build_lstm(input_dim, output_dim):
    model = models.Sequential([
        layers.Reshape((1, input_dim), input_shape=(input_dim,)),
        layers.LSTM(32),
        layers.Dense(output_dim, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='kl_divergence')
    return model

In [5]:
# ---------------------------------------------------------
# 4. 학습 및 미지 시료 추정
# ---------------------------------------------------------
model_dnn = build_dnn(X_train_scaled.shape[1], y_train.shape[1])
model_dnn.fit(X_train_scaled, y_train, epochs=50, batch_size=32, verbose=0)

# 미지 시료 예측 예시 (현장 측정 데이터 가정)
unknown_raw = np.array([[12.5, 3.2, 85, 12]]) 
unknown_scaled = scaler.transform(unknown_raw)
prediction = model_dnn.predict(unknown_scaled)

print("오염원별 추정 기여율:")
sources = ["Manure", "Sewage", "Fertilizer", "Soil", "Rain"]
for s, p in zip(sources, prediction[0]):
    print(f"{s}: {p*100:.2f}%")

1/1 [==============================] - 0s 68ms/step
오염원별 추정 기여율:
Manure: 31.87%
Sewage: 59.72%
Fertilizer: 3.58%
Soil: 2.60%
Rain: 2.23%


In [ ]:
# ---------------------------------------------------------
# 4. 학습 및 실제 미지 시료(CSV 파일) 추정
# ---------------------------------------------------------
# 1) 모델 학습 (DNN 예시)
model_dnn = build_dnn(X_train_scaled.shape[1], y_train.shape[1])
model_dnn.fit(X_train_scaled, y_train, epochs=50, batch_size=32, verbose=0)

# 2) CSV 파일 불러오기 (pandas 활용)
# 워킹 디렉토리에 'unknown_data.csv' 파일이 있다고 가정합니다.
# 실제 파일 경로로 수정하여 사용하세요. (예: "C:/data/unknown_data.csv")
unknown_samples_df = pd.read_csv("unknown_data.csv")

# 3) 분석에 필요한 특성(Feature)만 추출하여 Numpy 배열로 변환
# ★중요: 학습 데이터(X_train)에 사용된 컬럼명과 순서가 완벽히 일치해야 합니다.
feature_columns = ['d15N', 'd18O', 'Cl', 'TOC']
unknown_raw = unknown_samples_df[feature_columns].values

# 4) 데이터 스케일링
# ★핵심: 모델 학습 시 사용했던 동일한 scaler 객체의 'transform' 메서드를 사용해야 합니다. 
# (새롭게 fit_transform을 적용하면 스케일링 기준이 달라져 예측이 어긋납니다.)
unknown_scaled = scaler.transform(unknown_raw)

# 5) 딥러닝 모델로 기여율 예측
predictions = model_dnn.predict(unknown_scaled)

# 6) 예측 결과를 DataFrame으로 변환 및 원본 데이터와 병합
sources = ["Manure", "Sewage", "Fertilizer", "Soil", "Rain"]
# 컬럼명에 'p_'를 붙여 기여율(proportion)임을 명시
predictions_df = pd.DataFrame(predictions, columns=[f"p_{s}" for s in sources])

# 원본 DataFrame과 예측된 기여율 DataFrame을 가로(열 방향)로 병합
final_output_df = pd.concat([unknown_samples_df, predictions_df], axis=1)

# 결과 확인 (상위 5개 행 출력)
print("\n=== [실제 미지 시료 오염원 기여율 추정 결과 (DNN)] ===")
# 소수점 4자리까지 출력하도록 설정
print(final_output_df.round(4).head())

# 7) 최종 결과를 다시 CSV 파일로 내보내기 (선택 사항)
# final_output_df.to_csv("final_prediction_results_dnn.csv", index=False, encoding='utf-8-sig')